# 02 - BERT/Sentence-Transformer Embeddings

Run this after notebook 01. It loads `cleaned_resumeJD_pairs.csv`, creates resume/JD embeddings, evaluates cosine similarity against `match_score`, and saves `resume_embeddings.pkl`.

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

In [ ]:
DATA_PATH_CANDIDATES = [
    'cleaned_resumeJD_pairs.csv',
    '/content/cleaned_resumeJD_pairs.csv',
    '/content/drive/MyDrive/cleaned_resumeJD_pairs.csv',
]

data_path = next((p for p in DATA_PATH_CANDIDATES if os.path.exists(p)), None)
if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = next(iter(uploaded.keys()))
    except Exception as exc:
        raise FileNotFoundError('Upload cleaned_resumeJD_pairs.csv from notebook 01.') from exc

df = pd.read_csv(data_path)
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print('\nLabel distribution:')
print(df['match_label'].value_counts())
display(df.head(3))

In [ ]:
# Same style as the original notebook. This model creates 768-dimensional embeddings.
MODEL_NAME = 'all-mpnet-base-v2'
model = SentenceTransformer(MODEL_NAME)
print(f'Loaded model: {MODEL_NAME}')

In [ ]:
resume_texts = df['resume_text'].astype(str).tolist()
jd_texts = df['job_description'].astype(str).tolist()

resume_embeddings = model.encode(resume_texts, batch_size=16, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
jd_embeddings = model.encode(jd_texts, batch_size=16, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

print(f'resume_embeddings: {resume_embeddings.shape}')
print(f'jd_embeddings:     {jd_embeddings.shape}')

In [ ]:
df['bert_similarity'] = np.sum(resume_embeddings * jd_embeddings, axis=1)
df['error'] = (df['match_score'] - df['bert_similarity']).abs()

mae = mean_absolute_error(df['match_score'], df['bert_similarity'])
rmse = mean_squared_error(df['match_score'], df['bert_similarity']) ** 0.5
corr = df[['match_score', 'bert_similarity']].corr().iloc[0, 1]

print(f'MAE:         {mae:.4f}')
print(f'RMSE:        {rmse:.4f}')
print(f'Correlation: {corr:.4f}')

print('\nAverage similarity by label:')
display(df.groupby('match_label')[['match_score', 'bert_similarity', 'error']].mean().round(4))

print('\nTop 5 worst predictions:')
display(df.nlargest(5, 'error')[['match_label', 'match_score', 'bert_similarity', 'error']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['match_score'], df['bert_similarity'], alpha=0.65)
axes[0].plot([0, 1], [0, 1], 'r--')
axes[0].set_xlabel('Original match_score')
axes[0].set_ylabel('Embedding cosine similarity')
axes[0].set_title('Score vs Embedding Similarity')

for label, color in [('low', 'red'), ('medium', 'orange'), ('high', 'green')]:
    subset = df[df['match_label'] == label]['bert_similarity']
    axes[1].hist(subset, bins=20, alpha=0.6, label=label, color=color)
axes[1].set_title('BERT Similarity by Label')
axes[1].set_xlabel('bert_similarity')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('resume_embeddings_with_similarity.csv', index=False)

with open('resume_embeddings.pkl', 'wb') as f:
    pickle.dump({
        'model_name': MODEL_NAME,
        'resume_embeddings': resume_embeddings,
        'jd_embeddings': jd_embeddings,
        'bert_similarity': df['bert_similarity'].tolist(),
        'match_scores': df['match_score'].tolist(),
        'match_labels': df['match_label'].tolist(),
    }, f)

print('Saved: resume_embeddings.pkl')
print('Saved: resume_embeddings_with_similarity.csv')

try:
    from google.colab import files
    files.download('resume_embeddings.pkl')
    files.download('resume_embeddings_with_similarity.csv')
except Exception:
    pass